# 02 — Implementation of compact loop to run multiple models over dataset

This notebook reads the preprocessed dataset **`spotify_clean.csv`** and trains the following models:
- **Random Forest** (shown as `"tree"`)
- **XGBoost** (`"xgb"`)
- **KNN** (`"knn"`)
- **Gaussian Naive Bayes** (`"bayes"`)

Each model has different input assumptions, so we tailor preprocessing accordingly:
- **Trees / XGBoost:** order-based splits → no scaling required; minimal transforms (e.g., `log1p(duration_ms)`), numeric-only features engineered via `BaseFeatureEngineer` (e.g., `age` from `year`, cyclic `key`, optional `prod_intensity`).
- **KNN:** distance-based → **StandardScaler** plus light de-skewing (e.g., `log1p(duration_ms)`).
- **GaussianNB:** assumes per-class Gaussian features → **QuantileTransformer(output_distribution="normal")** after numeric feature engineering.

We convert `track_popularity` to **three tertiles** using **train-only** thresholds (to avoid leakage), run **StratifiedKFold** cross-validation with **GridSearchCV**, and evaluate on a held-out test set. We provide two optimization modes:
1) **High-class focus** (`use_f1_high=True`): refit on F1 for the “High” tertile (class 2).
2) **Balanced performance** (`use_f1_high=False`): refit on macro-F1 across all tertiles.

Diagnostics include **confusion matrices** (percent and counts) and, when available, **probability threshold curves** for the High class.  
Finally, we also run **tree-based models** (XGBoost and Random Forest) with a **one-hot-encoded** version of `artist_genres` (`"xgb_ohe"`, `"tree_ohe"`) to test whether genre signals improve performance.


In [ ]:
import pandas as pd

df = pd.read_csv("../data/spotify_clean.csv")  # now df holds your data


In [ ]:
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.preprocessing import MultiLabelBinarizer
from __future__ import annotations
import numpy as np
import pandas as pd
from typing import Tuple, Dict, Any

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import (
    StandardScaler, QuantileTransformer, FunctionTransformer, KBinsDiscretizer
)
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier


np.random.seed(42)

**MultiHotGenre**  
Turn a messy `artist_genres` column (lists, stringified lists like `['pop','dance']`, or delimited strings like `pop;dance|edm`) into a **multi-hot numeric matrix** usable by models.

**Key parameters**
- **`sep`**: main delimiter to split plain strings (also accepts `,` and `|`).
- **`min_freq`**: keep only genres that appear in at least this many **rows**.
- **`lowercase`**: optionally lowercase tokens so `Pop` = `pop`.


In [ ]:
import ast, re

class MultiHotGenre(BaseEstimator, TransformerMixin):
    def __init__(self, sep=";", min_freq=20, lowercase=False):
        self.sep = sep
        self.min_freq = min_freq
        self.lowercase = lowercase

    def _to_series(self, X):
        if isinstance(X, pd.DataFrame):
            s = X.iloc[:, 0]
        elif isinstance(X, pd.Series):
            s = X
        else:
            s = pd.Series(np.asarray(X).ravel())
        return s.fillna("")

    def _tokenize(self, v):
        # Accept real lists, stringified lists, or delimited strings
        if isinstance(v, (list, tuple, set)):
            toks = list(v)
        else:
            s = str(v).strip()
            # try to parse "['a','b']" etc.
            try:
                obj = ast.literal_eval(s)
                if isinstance(obj, (list, tuple, set)):
                    toks = list(obj)
                else:
                    toks = [obj]
            except Exception:
                s = s.strip().strip("[]")
                toks = re.split(rf"[{re.escape(self.sep)},|]", s)

        # normalise + dequote
        out = []
        for t in toks:
            t = str(t).strip().strip("'\"")
            if self.lowercase:
                t = t.lower()
            if t:
                out.append(t)
        return out

    def fit(self, X, y=None):
        s = self._to_series(X)
        lists = s.apply(self._tokenize)

        counts = {}
        for lst in lists:
            for g in set(lst):  # dedupe within row
                counts[g] = counts.get(g, 0) + 1

        self.keep_ = sorted([g for g, c in counts.items() if c >= self.min_freq])
        self.mlb_ = MultiLabelBinarizer(classes=self.keep_)
        self.mlb_.fit([[]])  # establish classes_
        self.out_feature_names_ = np.array([f"genre={g}" for g in self.keep_], dtype=object)
        # print("MultiHotGenre: kept", len(self.keep_), "genres")
        return self

    def transform(self, X):
        s = self._to_series(X)
        lists = s.apply(self._tokenize).apply(lambda lst: [g for g in lst if g in self.keep_]).tolist()
        if not self.keep_:
            return np.zeros((len(lists), 0), dtype=np.float32)
        return self.mlb_.transform(lists).astype(np.float32)

    def get_feature_names_out(self, input_features=None):
        return self.out_feature_names_


**BaseFeatureEngineer — purpose (numeric-only preprocessing)**  
Turns raw columns into a clean numeric feature set that’s model-friendly:

- **Keep numeric-only:** takes the input, keeps only numeric columns.

- **Converts column “year” to “age”:** creates `age = snapshot_year − year` (`snapshot_year` = max year in training set), then drops `year`.

- **Encodes musical “key” into `key_sin`/`key_cos`:** (so 11 continues to 0), then drops the original `key`.

- **“energy” and “loudness” are combined into `prod_intensity` (if enabled):** z-scores each on train, averages them into one feature, then drops the originals.

- **Adds `log_duration_ms = log1p(duration_ms)` and drops `duration_ms`.**

- **Final pruning:** drops a few low/less-useful columns: `instrumentalness`, `mode`, `key_sin`, `key_cos`.


**build_data_and_pipeline — what it wires up**  
- **Target**: bins `track_popularity` into **train-only** tertiles (avoids leakage).
- **Split**: train/test split before binning.
- **Preprocessing + model (by `modelName`)**  
  - `tree` / `xgb`: BaseFeatureEngineer → (optionally) genre multi-hot → RandomForest / XGBoost (no scaling).  
  - `knn`: BaseFeatureEngineer → `StandardScaler` → KNN (K needs scaling).  
  - `bayes`: BaseFeatureEngineer → `QuantileTransformer(output_distribution="normal")` → GaussianNB (Gaussianizes features; no extra scaling).
- **Grids**: returns a sensible param grid + pipeline + discretizer.


In [ ]:
# ---------- Custom transformers (DataFrame in/out) ----------

class BaseFeatureEngineer(BaseEstimator, TransformerMixin):
    """
    - Keep numerics only
    - age = snapshot_year - year  (drop raw 'year')
    - key -> (sin, cos), drop 'key'
    - prod_intensity = mean(zscore(energy), zscore(loudness)), drop originals
    - duration_ms -> log1p(duration_ms)
    """
    def __init__(self, combine_energy_loudness=True):
        self.combine_energy_loudness = combine_energy_loudness

    def fit(self, X, y=None):
        X = pd.DataFrame(X)
        self._has_year_ = "year" in X.columns
        if self._has_year_:
            self.snapshot_year_ = int(X["year"].max())
        else:
            self.snapshot_year_ = None

        self._can_combine_ = (
            self.combine_energy_loudness and {"energy", "loudness"}.issubset(X.columns)
        )
        if self._can_combine_:
            self.en_mean_ = float(X["energy"].mean())
            self.en_std_  = float(X["energy"].std(ddof=0) or 1.0)
            self.loud_mean_ = float(X["loudness"].mean())
            self.loud_std_  = float(X["loudness"].std(ddof=0) or 1.0)

        # compute and store output feature names
        cols = self.transform(X).columns.tolist()
        self.out_feature_names_ = np.array(cols, dtype=object)
        return self

    def get_feature_names_out(self, input_features=None):
        return getattr(self, "out_feature_names_", np.array([], dtype=object))

    def transform(self, X):
        df = pd.DataFrame(X).select_dtypes(include=[np.number]).copy()

        # age from year
        if self._has_year_ and "year" in df.columns:
            df["age"] = self.snapshot_year_ - df["year"]
            df.drop(columns=["year"], inplace=True, errors="ignore")

        # key cyclic
        if "key" in df.columns:
            ang = 2 * np.pi * df["key"].astype(float) / 12.0
            df["key_sin"] = np.sin(ang)
            df["key_cos"] = np.cos(ang)
            df.drop(columns=["key"], inplace=True, errors="ignore")

        # combine energy + loudness into one feature
        if self._can_combine_ and {"energy", "loudness"}.issubset(df.columns):
            e_z = (df["energy"] - self.en_mean_) / (self.en_std_ or 1.0)
            l_z = (df["loudness"] - self.loud_mean_) / (self.loud_std_ or 1.0)
            df["prod_intensity"] = (e_z + l_z) / 2.0
            df.drop(columns=["energy", "loudness"], inplace=True, errors="ignore")

        # log-transform duration_ms (replace original)
        if "duration_ms" in df.columns:
            df["log_duration_ms"] = np.log1p(df["duration_ms"])
            df.drop(columns=["duration_ms"], inplace=True, errors="ignore")


        # drop unwanted columns without enough variance or that didn't help with feature importance
        for c in ["instrumentalness", "mode", "key_sin", "key_cos"]:
            if c in df.columns:
                df.drop(columns=[c], inplace=True)

        return df


# ---------- Main helper ----------

def build_data_and_pipeline(
    df_clean: pd.DataFrame,
    modelName: str,
    test_size: float = 0.2,
    random_state: int = 42,
    use_genre_ohe: bool = False,   # <-- NEW
    genre_col: str = "artist_genres",      # <-- NEW
    genre_min_freq: int = 20       # <-- NEW
) -> Tuple[pd.DataFrame, pd.DataFrame, np.ndarray, np.ndarray, Pipeline, Dict[str, Any], KBinsDiscretizer]:
    """
    Returns (X_train, X_test, y_train, y_test, pipeline, param_grid, discretizer)
    - Uses only numeric columns from df_clean
    - Creates tertiles for 'track_popularity' using TRAIN split only
    - modelName ∈ {'tree','knn','bayes', 'xgb'} selects preprocessing + estimator
    """
    assert modelName in {"tree","knn","bayes","xgb", "xgb_ohe", "tree_ohe"}, "modelName must be one of {'tree','knn','bayes','xgb'}"
    assert "track_popularity" in df_clean.columns, "df_clean must contain 'track_popularity'"

    target = "track_popularity"
    X = df_clean.drop(columns=[target]).copy()
    y = df_clean[target].copy().astype(float).to_numpy()

    X_train, X_test, y_train_raw, y_test_raw = train_test_split(
        X, y, test_size=test_size, random_state=random_state
    )

    has_genre = use_genre_ohe and (genre_col in df_clean.columns)

    # tertiles from TRAIN only
    disc = KBinsDiscretizer(n_bins=3, encode="ordinal", strategy="quantile")
    y_train = disc.fit_transform(y_train_raw.reshape(-1,1)).astype(int).ravel()
    y_test  = disc.transform(y_test_raw.reshape(-1,1)).astype(int).ravel()

    # common base feature engineering
    base = BaseFeatureEngineer(
        combine_energy_loudness=True,
    )

    if modelName == "tree":
        base = BaseFeatureEngineer(
        combine_energy_loudness=False,
        )
        num_pipe = Pipeline([
            ("base", base)
        ])

        if has_genre:
            pre = ColumnTransformer(
                transformers=[
                    ("num", num_pipe, make_column_selector(dtype_include=np.number)),
                    ("cat", MultiHotGenre(sep=";", min_freq=genre_min_freq), [genre_col]),
                ],
                remainder="drop"
            )
        else:
            pre = num_pipe

        clf = RandomForestClassifier(random_state=random_state, n_jobs=-1)
        grid = {
            "clf__n_estimators": [300, 600],
            "clf__max_depth": [None, 12, 20],
            "clf__min_samples_leaf": [1, 3, 10],
            "clf__max_features": ["sqrt", 0.5, None],
        }
        pipe = Pipeline([("pre", pre), ("clf", clf)])


    elif modelName == "knn":
        base = BaseFeatureEngineer(
        combine_energy_loudness=False,
        )
        # KNN: log1p(duration_ms) already in base; just scale
        pre = Pipeline([
            ("base", base),
            ("scale", StandardScaler()),
        ])
        clf = KNeighborsClassifier()
        grid = {
            "clf__n_neighbors": [5, 15, 35],
            "clf__weights": ["uniform", "distance"],
            "clf__p": [1, 2],  # controls Minkowski distance
        }
        pipe = Pipeline([("pre", pre), ("clf", clf)])

    elif modelName == "xgb":
        base = BaseFeatureEngineer(
        combine_energy_loudness=False,
        )
        # XGB: log1p(duration_ms) ixn base; no scaling
        num_pipe = Pipeline([
            ("base", base),
        ])

        if has_genre:
            pre = ColumnTransformer(
                transformers=[
                    ("num", num_pipe, make_column_selector(dtype_include=np.number)),
                    ("cat", MultiHotGenre(sep=";", min_freq=genre_min_freq), [genre_col]),
                ],
                remainder="drop"
            )
        else:
            pre = num_pipe

        # XGBoost is building up to 600 depth-6 trees, each trained on 80% of rows and 80% of features, with a modest learning rate (0.1) and some regularisation (min_child_weight, reg_lambda), optimising a multiclass probabilistic loss (multi:softprob / mlogloss).
        clf = XGBClassifier(
            objective="multi:softprob", # multiclass objective
            num_class=3,
            n_estimators=600,
            max_depth=6,
            learning_rate=0.1,
            subsample=0.8, # Fraction of training rows sampled (without replacement) for each tree. acts like bagging if <1
            colsample_bytree=0.8, # Fraction of features randomly sampled for each tree (Like Random Forest’s max_features -> decorrelates trees, improves robustness)
            min_child_weight=1.0, # the lower the smaller leaves are allowed.
            reg_lambda=1.0, # Larger values penalise large leaf scores -> smoother, less extreme predictions
            eval_metric="mlogloss", # Evaluation metric for training logs / early stopping
            tree_method="hist", # Uses histogram-based algorithm for building trees
            n_jobs=-1,# uses all available CPUs
            random_state=random_state,
        )
        grid = {
            "clf__n_estimators": [300, 600, 900],
            "clf__max_depth": [3, 6],
            "clf__learning_rate": [0.05, 0.1],
            "clf__subsample": [0.8, 1.0],
            "clf__colsample_bytree": [0.8, 1.0],
            "clf__min_child_weight": [1, 5],
        }
        pipe = Pipeline([("pre", pre), ("clf", clf)])

    else:  # "bayes"
        # GaussianNB: log1p(duration_ms) in base; then Gaussianize
        # First, transform each feature to look roughly Gaussian; then fit a Gaussian Naive Bayes, and tune how smooth its Gaussians are.”
        pre = Pipeline([
            ("base", base),
            ("gauss", QuantileTransformer(
                output_distribution="normal",
                random_state=random_state
            )),
        ])

        # for each feature:
        # Before: “weird shape” (skewed, bounded, etc.)
        # After: approximately N(0,1) overall.
        clf = GaussianNB()
        grid = {"clf__var_smoothing": [1e-9, 1e-8, 1e-7, 1e-6]} # var_smoothing adds a tiny bit to each variance (acts as a regularisation on the Gaussian widths)
        pipe = Pipeline([("pre", pre), ("clf", clf)])

        # We’re using GaussianNB knowing that the independence assumption is only approximately true and that some features remain correlated. The QuantileTransformer makes the Gaussian assumption more reasonable; the independence violation is handled pragmatically (and empirically the classifier still performs well).

    return X_train, X_test, y_train, y_test, pipe, grid, disc


**Two helper functions for SHAP visualisations for tree models (used only for Random Forests "tree" models)**

- `plot_shap_bar_for_class(...)`: For a chosen class (e.g., **Low/Medium/High**), it:
  1) extracts the trained pipeline’s preprocessor and classifier,  
  2) computes SHAP values,  
  3) ranks features by average absolute contribution (importance),  
  4) plots a green/red bar chart of the **signed mean SHAP** per feature  
     (green ↑ increases that class, red ↓ decreases it).

- `shap_global_bar(...)`: Shows **global SHAP importance** (mean |SHAP|) across classes/samples as a simple bar plot.  
  If there are 3 classes, it labels them **Low / Medium / High**.

These plots explain which features drive the model’s predictions and in which direction; they are designed for **tree-based models** (e.g., Random Forest, XGBoost).


In [ ]:
def plot_shap_bar_for_class(best_model, X_test, class_index=0, class_name=None, top_n=20, rank_by="abs"):
    import numpy as np, pandas as pd, matplotlib.pyplot as plt
    import shap

    # pull pre + clf
    pre = getattr(best_model, "named_steps", {}).get("pre", None)
    clf = getattr(best_model, "named_steps", {}).get("clf", best_model)

    # preprocess + names
    X_pre = pre.transform(X_test) if pre is not None else X_test
    try:
        feat = list(pre.get_feature_names_out()) if pre is not None else (list(X_pre.columns) if hasattr(X_pre,"columns") else None)
    except Exception:
        feat = list(X_pre.columns) if hasattr(X_pre,"columns") else None
    if feat is None: feat = [f"f{i}" for i in range(X_pre.shape[1])]
    X_df = X_pre if hasattr(X_pre,"columns") else pd.DataFrame(X_pre, columns=feat)

    # SHAP values
    sv = shap.Explainer(clf, X_df)(X_df).values  # (n,f) or (n,c,f)

    # pick class matrix
    M = sv[:, class_index, :] if sv.ndim == 3 else (sv if class_index == 1 else -sv)

    mean_signed = M.mean(axis=0)
    mean_abs = np.abs(M).mean(axis=0)
    order = np.argsort(mean_abs if rank_by == "abs" else np.abs(mean_signed))[::-1][:top_n]

    names = np.array(feat)[order]
    vals  = mean_signed[order]
    colors = ['#2ecc71' if v > 0 else '#e74c3c' for v in vals]
    title = class_name or f"Class {class_index}"

    plt.figure(figsize=(12, 0.45*len(order)+2))
    plt.barh(range(len(order)), vals[::-1], color=colors[::-1], alpha=0.85)
    plt.yticks(range(len(order)), names[::-1])
    plt.xlabel(f'Average contribution towards {title}')
    plt.title(f'Top {len(order)} Variables: Direction of Contribution towards {title}\n'
              '(Green = Increases | Red = Decreases)')
    plt.axvline(0, color='black', ls='--', lw=1, alpha=0.6)
    plt.grid(axis='x', alpha=0.25)
    plt.tight_layout()
    plt.show()

def shap_global_bar(best_model, X_test, max_display=20):
    import pandas as pd, shap
    # pull pre + clf
    pre = getattr(best_model, "named_steps", {}).get("pre", None)
    clf = getattr(best_model, "named_steps", {}).get("clf", best_model)

    # preprocess + names
    X_pre = pre.transform(X_test) if pre is not None else X_test
    try:
        feat = list(pre.get_feature_names_out()) if pre is not None else (list(X_pre.columns) if hasattr(X_pre,"columns") else None)
    except Exception:
        feat = list(X_pre.columns) if hasattr(X_pre,"columns") else None
    if feat is None: feat = [f"f{i}" for i in range(X_pre.shape[1])]
    X_df = X_pre if hasattr(X_pre,"columns") else pd.DataFrame(X_pre, columns=feat)

    expl = shap.Explainer(clf, X_df)
    exp  = expl(X_df)  # exp.values: (n,f) or (n,c,f)
    # class names for multiclass
    class_names = ["Low","Medium","High"]# if getattr(exp.values, "ndim", 2) == 3 and exp.values.shape[1] == 3 else None

    shap.summary_plot(exp.values, X_df, plot_type="bar", max_display=max_display, class_names=class_names, show=True)


**Purpose — training+evaluation harness (`run_model`)**

- Wraps a model-specific pipeline (built by `build_data_and_pipeline`) in **GridSearchCV** with **StratifiedKFold** and evaluates on a held-out test set.
- Supports optional **one-hot genre** features for tree/XGB models via name suffix `_ohe` (e.g., `"tree_ohe"`, `"xgb_ohe"`).

**Custom pieces**

- `f1_high_only(y_true, y_pred)`: computes **F1** treating class **2 (“High”)** as the positive class (one-vs-rest).
- `_split_model_and_ohe(name)`: parses `modelName` and toggles genre OHE only for `tree`/`xgb` variants ending in `_ohe`.

**Cross-validation & scoring**

- CV: `StratifiedKFold(n_splits=5, shuffle=True)`.
- Scoring dict includes:  
  - `accuracy`, `f1_macro` (balanced across tertiles), and `f2_macro` (recall-leaning).  
  - If `use_f1_high=True`, adds `f1_high` and **refits** the grid on `f1_high`; otherwise refits on `f1_macro`.

**What gets tuned**

- The **pipeline** (`pipe`) and its **param grid** (`param_grid`) come from `build_data_and_pipeline` (depends on `modelName`: `tree`, `xgb`, `knn`, `bayes`, and their `_ohe` variants).

**Reporting**

- Prints best params and **CV means** (for all tracked metrics at the chosen hyperparameters).
- Computes **test metrics**: `accuracy`, `f1_macro`, and if enabled: **precision/recall/F1 for “High” class (2)**.

**Confusion matrices**

- Plots **row-normalized** confusion matrix (% of actual class) and **raw counts** version for classes `[Low, Medium, High]` = labels `[0,1,2]`.

**Threshold curves (High class)**

- If the estimator supports `predict_proba`, sweeps thresholds on `P(class=2)` to plot **precision/recall/F1 vs threshold** (baseline marker at 0.5).

**Feature importance (tree/XGB)**

- If the classifier has `feature_importances_`, extracts **feature names** (tries `pre.get_feature_names_out()`; otherwise infers) and plots top importances.

**SHAP explanations (optional)**

- When `print_shap=True` **and** `base_model == 'tree'`:
  - **Global SHAP bar**: mean \|SHAP\| per feature across samples (and classes), shown as a bar chart (labels **Low/Medium/High** if 3 classes).
  - **Per-class SHAP bars**: for each class, plots a green/red bar chart of **signed mean SHAP** (direction) ranked by **mean \|SHAP\|** (importance).  
    Green → increases that class’s score; Red → decreases.

**Return value**

- Dict with: `best_model`, `best_params`, CV scores at the chosen params, test metrics, and the test/predicted labels.


In [ ]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    make_scorer,
    fbeta_score,
    ConfusionMatrixDisplay,
    precision_score,
    recall_score,
    confusion_matrix,
)
import matplotlib.pyplot as plt
import numpy as np


# custom metric used to refit grid search on combination that maximises f1 for class High specifically
def f1_high_only(y_true, y_pred):
    """F1 score treating class 2 (High) as the positive class."""
    y_true_h = (y_true == 2).astype(int)
    y_pred_h = (y_pred == 2).astype(int)
    return f1_score(y_true_h, y_pred_h, zero_division=0)


# helper function to decide whether to ohe or not depending on whether the model name ends in "_ohe"). Intendedlly only works for tree and xgb
def _split_model_and_ohe(name: str):
    use_ohe = name.endswith("_ohe")
    base = name[:-4] if use_ohe else name
    if use_ohe and base not in {"tree", "xgb"}:
        # only tree models support OHE here
        use_ohe = False
    return base, use_ohe


def run_model(modelName, df, use_f1_high=True, test_size=0.2, random_state=42, cv_splits=5, print_shap=False):
    base_model, want_ohe = _split_model_and_ohe(modelName)
    X_train, X_test, y_train, y_test, pipe, param_grid, disc = build_data_and_pipeline(
        df,
        modelName=base_model,  # <-- use base ('xgb' or 'tree' etc.)
        test_size=test_size,
        random_state=random_state,
        use_genre_ohe=want_ohe,  # <-- turn on OHE if *_ohe
        genre_col="artist_genres",
        genre_min_freq=20,
    )

    cv = StratifiedKFold(n_splits=cv_splits, shuffle=True, random_state=random_state)

    # Scoring: always log accuracy + F1; optionally add F2 and refit on it
    # we use macro averaging because all classes have the same weights
    scoring = {
        "accuracy": "accuracy",
        "f1_macro": "f1_macro",
        "f2_macro": make_scorer(fbeta_score, beta=2, average="macro"),
    }
    # we keep track of f2_macro but we do prefer f1_macro (false positives are more costly than false negatives)
    refit_metric = "f1_macro"
    if use_f1_high:
        scoring["f1_high"] = make_scorer(f1_high_only)
        refit_metric = "f1_high"

    grid = GridSearchCV(
        pipe,
        param_grid,
        cv=cv,
        scoring=scoring,
        refit=refit_metric,
        n_jobs=-1,
        verbose=1,
    )
    grid.fit(X_train, y_train)
    best_model = grid.best_estimator_
    pred = best_model.predict(X_test)

    # Print summary
    print(f"\n=== {modelName.upper()} (refit on {refit_metric}) ===")
    print("Best params:", grid.best_params_)
    print(
        "CV means at chosen params:",
        {m: grid.cv_results_[f"mean_test_{m}"][grid.best_index_] for m in scoring},
    )

    test_metrics = {
        "accuracy": accuracy_score(y_test, pred),
        "f1_macro": f1_score(y_test, pred, average="macro"),
    }
    if use_f1_high:
        # High-class (2) one-vs-rest metrics
        y_true_high = (y_test == 2).astype(int)
        y_pred_high = (pred == 2).astype(int)
        test_metrics["precision_high"] = precision_score(y_true_high, y_pred_high, zero_division=0)
        test_metrics["recall_high"] = recall_score(y_true_high, y_pred_high, zero_division=0)
        test_metrics["f1_high"] = f1_score(y_true_high, y_pred_high, zero_division=0)
    print("Test metrics:", test_metrics)

    print("\nClassification report:")
    print(classification_report(y_test, pred, labels=[0, 1, 2], target_names=["Low", "Medium", "High"]))

    # % confusion matrix (row-normalized)
    ConfusionMatrixDisplay.from_predictions(
        y_test,
        pred,
        labels=[0, 1, 2],
        display_labels=["Low", "Medium", "High"],
        normalize="true",
        values_format=".1%",
        cmap="Blues",
        colorbar=False,
    )
    plt.title(f"{modelName.upper()} — Confusion Matrix (% of actual class)")
    plt.tight_layout()
    plt.show()

    # Confusion matrix with raw counts
    print("\nConfusion matrix (counts):")
    ConfusionMatrixDisplay.from_predictions(
        y_test,
        pred,
        labels=[0, 1, 2],
        display_labels=["Low", "Medium", "High"],
        normalize=None,  # <-- raw counts
        values_format="d",
        cmap="Blues",
        colorbar=False,
    )
    plt.title(f"{modelName.upper()} — Confusion Matrix (counts)")
    plt.tight_layout()
    plt.show()

    # ===== Threshold curves for the High class (2) =====
    if hasattr(best_model, "predict_proba"):
        y_true_high = (y_test == 2).astype(int)
        proba = best_model.predict_proba(X_test)
        scores_high = proba[:, 2]
        thresholds = np.linspace(0.0, 0.9, 10)
        precs, recs, f1s = [], [], []
        for t in thresholds:
            y_pred_high = (scores_high >= t).astype(int)
            precs.append(precision_score(y_true_high, y_pred_high, zero_division=0))
            recs.append(recall_score(y_true_high, y_pred_high, zero_division=0))
            f1s.append(f1_score(y_true_high, y_pred_high, zero_division=0))
        plt.figure()
        plt.plot(thresholds, recs, label="Recall @ t")
        plt.plot(thresholds, precs, label="Precision @ t")
        plt.plot(thresholds, f1s, label="F1 @ t")
        plt.axvline(x=0.5, linestyle="--", color="grey", label="t = 0.5")
        plt.xlabel("Threshold on P(High)")
        plt.ylabel("Metric @ Threshold")
        plt.title(f"{modelName.upper()} — High class metrics vs threshold")
        plt.legend()
        plt.tight_layout()
        plt.show()
    else:
        print("Estimator has no predict_proba; skipping threshold plot.")

    # ===== Feature importances (for RF / XGB, etc.) =====
    # try to pull out preprocessor + classifier from the pipeline
    clf = None; pre = None
    try:
        pre = best_model.named_steps.get("pre", None)
        clf = best_model.named_steps.get("clf", None)
    except AttributeError:
        clf = best_model

    if clf is not None and hasattr(clf, "feature_importances_"):
        # <-- NEW: try to get names from the preprocessor itself
        feature_names = None
        if pre is not None:
            try:
                feature_names = pre.get_feature_names_out()
            except Exception:
                pass

        if feature_names is None:
            X_train_pre = pre.transform(X_train) if pre is not None else X_train
            feature_names = (list(X_train_pre.columns)
                             if hasattr(X_train_pre, "columns")
                             else [f"f{i}" for i in range(X_train_pre.shape[1])])

        importances = clf.feature_importances_
        idx = np.argsort(importances)[::-1]
        top_n = 6# min(20, len(idx))
        plt.figure(figsize=(8, 0.4 * top_n + 3))
        plt.barh(range(top_n), importances[idx[:top_n]][::-1])
        plt.yticks(range(top_n), [feature_names[i] for i in idx[:top_n]][::-1])
        plt.xlabel("Feature importance")
        plt.title(f"{modelName.upper()} — Feature importances")
        plt.tight_layout()
        plt.show()
    else:
        print("No feature_importances_ attribute; skipping feature-importance plot.")

    if base_model == 'tree' and print_shap==True:
        #SHAP analysis
        shap_global_bar(best_model, X_test, max_display=20)
        # ----- Simple SHAP bars per class -----
        try:
            # number of classes (prefer proba if available)
            if hasattr(best_model, "predict_proba"):
                n_classes = best_model.predict_proba(X_test).shape[1]
            else:
                n_classes = len(np.unique(y_test))

            class_names = ["Low", "Medium", "High"] if n_classes == 3 else [f"Class {i}" for i in range(n_classes)]

            # Plot one figure per class
            for c in range(n_classes):
                plot_shap_bar_for_class(best_model, X_test, class_index=c, class_name=class_names[c], top_n=20)
        except Exception as e:
            print(f"SHAP per-class plots skipped: {e}")

    # ---- Export preprocessed dataset for external calibration (tree_ohe only) ----
    # if modelName == "tree_ohe" and use_f1_high == True:
    #     pre = best_model.named_steps.get("pre", None)
    #     if pre is not None:
    #         Xtr = pre.transform(X_train); Xte = pre.transform(X_test)
    #         feat_names = pre.get_feature_names_out()
    #         import pandas as pd
    #         Xtr = pd.DataFrame(Xtr, columns=feat_names); Xtr.insert(0, "target", y_train); Xtr["split"] = "train"
    #         Xte = pd.DataFrame(Xte, columns=feat_names); Xte.insert(0, "target", y_test);  Xte["split"] = "test"
    #         pd.concat([Xtr, Xte], ignore_index=True).to_csv("../data/spotify_clean_tree_ohe.csv", index=False)


    return {
        "modelName": modelName,
        "best_model": best_model,
        "best_params": grid.best_params_,
        "cv": {m: grid.cv_results_[f"mean_test_{m}"][grid.best_index_] for m in scoring},
        "test": test_metrics,
        "y_test": y_test,
        "pred": pred,
    }


**Loop 1 — optimize for “High” class:**  
Runs each model variant and refits on **F1 for the High class (class 2)** via `use_f1_high=True`.

In [2]:
for modelName in ["tree", "xgb", "knn", "bayes", "xgb_ohe", "tree_ohe"]:
    run_model(modelName, df, use_f1_high=True)

    # Use this line instead if you want to view SHAP analysis for Random Forest "tree" models
    # run_model(modelName, df, use_f1_high=True, print_shap=True)



**Loop 2 — balanced performance:**  
Runs each model variant and refits on **macro-F1 across all tertiles** via `use_f1_high=False`.


In [1]:
for m in ["xgb", "knn", "tree", "bayes", "xgb_ohe", "tree_ohe"]:
    run_model(m, df, use_f1_high=False)

    # Use this line instead if you want to view SHAP analysis for Random Forest "tree" models
    # run_model(modelName, df, use_f1_high=False, print_shap=True)